# Step 2: CV Ranking & Evaluation

This notebook builds on **classify.ipynb** (Step 1) to:
1. **Rank multiple CVs** against a job description (JD) using the trained classifier + semantic similarity.
2. **Evaluate the matcher** without ground-truth match labels:
   - Same-domain vs different-domain scores (CV and JD from same role should score higher).
   - Ranking accuracy: correct JD in top-k for each CV.

**Prerequisites:** Run `classify.ipynb` first so that `./cv_classifier_model` and data files exist.

## 1. Load saved classifier and label encoder

In [1]:
import json
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

C:\Users\dehem\anaconda3\envs\projectdata\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Rebuild label encoder (same as in classify.ipynb)
with open('data.json', 'r') as f:
    dataset_json = json.load(f)
labels_raw = [item["label"] for item in dataset_json]
le = LabelEncoder()
le.fit_transform(labels_raw)
print("Classes:", list(le.classes_))

Classes: [np.str_('EXPERIENCE'), np.str_('QUALIFICATION'), np.str_('SKILL')]


In [3]:
# Load saved model and tokenizer from Step 1
model_name = "prajjwal1/bert-tiny"
tokenizer = AutoTokenizer.from_pretrained("./cv_classifier_model")
model = AutoModelForSequenceClassification.from_pretrained("./cv_classifier_model", num_labels=len(le.classes_))
model.eval()
print("Classifier loaded.")

Classifier loaded.


## 2. Helper: classify line and parse CV/JD into categories

In [4]:
LABELS = {0: "SKILL", 1: "QUALIFICATION", 2: "EXPERIENCE"}

def classify_line(line, tokenizer=tokenizer, model=model):
    if not line or len(line.strip()) < 2:
        return None
    inputs = tokenizer(line.strip(), return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax(dim=1).item()
    return LABELS[pred]

def parse_text_to_result(text):
    """Split text into lines and classify each; return dict with skills, qualifications, experience."""
    lines = [l.strip() for l in text.replace(",", "\n").split("\n") if len(l.strip()) > 2]
    result = {"skills": [], "qualifications": [], "experience": []}
    for line in lines:
        label = classify_line(line)
        if label == "SKILL":
            result["skills"].append(line)
        elif label == "QUALIFICATION":
            result["qualifications"].append(line)
        elif label == "EXPERIENCE":
            result["experience"].append(line)
    return result

## 3. Load embedding model and compute match score

In [5]:
from sentence_transformers import SentenceTransformer, util

WEIGHTS = {"skills": 0.4, "experience": 0.4, "qualification": 0.2}
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def get_match_score(result_cv, result_jd, method="max"):
    """
    method: 'max' = single best pair per category (as in classify.ipynb);
    'mean' or 'hybrid' possible if you use similarity_matcher.compute_match_score.
    """
    total = 0.0
    for cat, key_jd in [("skills", "skills"), ("qualifications", "qualifications"), ("experience", "experience")]:
        cv_list = result_cv.get(cat, [])
        jd_list = result_jd.get(key_jd, [])
        if not cv_list or not jd_list:
            continue
        cv_emb = embedding_model.encode(cv_list, convert_to_tensor=True)
        jd_emb = embedding_model.encode(jd_list, convert_to_tensor=True)
        sim = util.cos_sim(cv_emb, jd_emb).max().item()
        w = WEIGHTS["skills"] if cat == "skills" else (WEIGHTS["qualification"] if cat == "qualifications" else WEIGHTS["experience"])
        total += w * sim
    return total

## 4. CV ranking: rank multiple CVs for one job

In [ ]:
# Load CVs (data 02.csv has Category, Resume)
df_cvs = pd.read_csv('data 02.csv')
# Load jobs
with open('Dataset_jotpars.csv', 'rb') as f:
    text = f.read().decode('utf-8', errors='ignore')
with open('Dataset_jotpars_clean.csv', 'w', encoding='utf-8') as f:
    f.write(text)
df_job = pd.read_csv('Dataset_jotpars_clean.csv')
df_job['full job post'] = df_job['requirment'].fillna('') + ' ' + df_job['description'].fillna('')
print("CVs:", len(df_cvs), "| Jobs:", len(df_job))

In [ ]:
def rank_cvs_for_job(job_index, df_cvs=df_cvs, df_job=df_job, top_k=10):
    """For one job (by index), score all CVs and return top-k (highest score first)."""
    jd_text = df_job.iloc[job_index]['full job post']
    result_jd = parse_text_to_result(jd_text)
    scores_list = []
    for i, row in df_cvs.iterrows():
        result_cv = parse_text_to_result(row['Resume'])
        score = get_match_score(result_cv, result_jd)
        scores_list.append((i, row['Category'], score))
    scores_list.sort(key=lambda x: x[2], reverse=True)
    return scores_list[:top_k], result_jd

In [ ]:
# Example: rank CVs for job at index 4237 (or any index)
job_idx = 4237
top_cvs, _ = rank_cvs_for_job(job_idx, top_k=10)
job_title = df_job.iloc[job_idx]['name']
print(f"Job: {job_title}")
print("Top 10 CVs by match score:")
for rank, (idx, category, score) in enumerate(top_cvs, 1):
    print(f"  {rank}. {category} — {score*100:.2f}%")

## 5. Evaluation: same-domain vs different-domain

In [ ]:
# Map job title (name) to a category similar to CV categories
def job_name_to_category(name):
    n = (name or "").lower()
    if "frontend" in n or "ui" in n or "react" in n and "developer" in n:
        return "Frontend Developer"
    if "python" in n and "developer" in n:
        return "Python Developer"
    if "data scientist" in n or "data science" in n or "ai engineer" in n or "machine learning" in n:
        return "Data Scientist"
    if "full stack" in n:
        return "Full Stack Developer"
    if "backend" in n or ".net" in n or "java" in n or "node" in n:
        return "Backend Developer"
    return "Other"

In [ ]:
# Precompute JD results and job categories (sample for speed)
n_jobs_sample = 500
n_cvs_sample = 200
job_results = []
job_categories = []
for i in range(min(n_jobs_sample, len(df_job))):
    job_results.append(parse_text_to_result(df_job.iloc[i]['full job post']))
    job_categories.append(job_name_to_category(df_job.iloc[i]['name']))
cv_results = []
cv_categories = []
for i in range(min(n_cvs_sample, len(df_cvs))):
    cv_results.append(parse_text_to_result(df_cvs.iloc[i]['Resume']))
    cv_categories.append(df_cvs.iloc[i]['Category'])
print("Precomputed", len(job_results), "jobs,", len(cv_results), "CVs.")

In [ ]:
# Same-domain: CV category == job category; different-domain: !=
same_scores, diff_scores = [], []
for i, (res_cv, cat_cv) in enumerate(zip(cv_results, cv_categories)):
    for j, (res_jd, cat_jd) in enumerate(zip(job_results, job_categories)):
        if cat_jd == "Other":
            continue
        score = get_match_score(res_cv, res_jd)
        if cat_cv == cat_jd:
            same_scores.append(score)
        else:
            diff_scores.append(score)
print("Mean score (same domain):", np.mean(same_scores))
print("Mean score (different domain):", np.mean(diff_scores))
print("Difference (same - different):", np.mean(same_scores) - np.mean(diff_scores))
print("Expected: same > different → matcher separates relevant from irrelevant pairs.")

## 6. Evaluation: ranking accuracy (correct JD in top-k)

In [ ]:
def ranking_accuracy_topk(cv_results, cv_categories, job_results, job_categories, k=1, n_cvs=50, jds_per_cv=5):
    """
    For each of n_cvs CVs: pick 1 same-domain JD and (jds_per_cv-1) different-domain JDs.
    Rank by score; count how often the same-domain JD is in top-k.
    """
    # Build list of job indices by category (excluding Other)
    from collections import defaultdict
    jobs_by_cat = defaultdict(list)
    for j, c in enumerate(job_categories):
        if c != "Other":
            jobs_by_cat[c].append(j)
    correct = 0
    total = 0
    for i in range(min(n_cvs, len(cv_results))):
        cat_cv = cv_categories[i]
        if cat_cv not in jobs_by_cat or not jobs_by_cat[cat_cv]:
            continue
        same_jd_idx = np.random.choice(jobs_by_cat[cat_cv])  # one correct JD
        other_cats = [c for c in jobs_by_cat if c != cat_cv and jobs_by_cat[c]]
        if len(other_cats) < jds_per_cv - 1:
            continue
        other_indices = []
        for _ in range(jds_per_cv - 1):
            c = np.random.choice(other_cats)
            other_indices.append(np.random.choice(jobs_by_cat[c]))
        candidate_indices = [same_jd_idx] + other_indices
        np.random.shuffle(candidate_indices)
        idx_correct = candidate_indices.index(same_jd_idx)
        scores = [get_match_score(cv_results[i], job_results[j]) for j in candidate_indices]
        top_k_idx = np.argsort(scores)[-k:].tolist()
        if idx_correct in top_k_idx:
            correct += 1
        total += 1
    return correct / total if total else 0.0, total

In [ ]:
np.random.seed(42)
acc_top1, n = ranking_accuracy_topk(cv_results, cv_categories, job_results, job_categories, k=1, n_cvs=80, jds_per_cv=5)
print(f"Top-1 ranking accuracy: {acc_top1*100:.1f}% (n={n} CVs)")
acc_top2, n2 = ranking_accuracy_topk(cv_results, cv_categories, job_results, job_categories, k=2, n_cvs=80, jds_per_cv=5)
print(f"Top-2 ranking accuracy: {acc_top2*100:.1f}% (n={n2} CVs)")

## 7. Optional: try a stronger embedding model

We can swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` (better quality, slower) and re-run the ranking and evaluation cells to compare.

In [ ]:
# Uncomment to try a stronger model:
# embedding_model = SentenceTransformer('all-mpnet-base-v2')
# Then re-run: rank_cvs_for_job(...), same-domain vs different-domain, and ranking_accuracy_topk.